In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import torch
import os
import numpy as np
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def remap_mask(mask):
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

class SUIMDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, image_folder, mask_folder, transform=None):
        self.root_dir = root_dir
        self.image_dir = os.path.join(root_dir, image_folder)
        self.mask_dir = os.path.join(root_dir, mask_folder)
        self.transform = transform

        self.image_filenames = sorted(os.listdir(self.image_dir))
        self.mask_filenames = sorted(os.listdir(self.mask_dir))

        if len(self.image_filenames) != len(self.mask_filenames):
            raise ValueError("Number of images and masks do not match!")

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_name = os.path.join(self.image_dir, self.image_filenames[idx])
        mask_name = os.path.join(self.mask_dir, self.mask_filenames[idx])

        image = Image.open(img_name).convert("RGB")
        mask = Image.open(mask_name).convert("L")

        image_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0])
        ])

        mask_transform = transforms.Compose([
            transforms.Resize((256, 256), interpolation=Image.NEAREST),
            transforms.Lambda(lambda img: torch.tensor(np.array(img), dtype=torch.long))
        ])

        image = image_transform(image)
        mask = mask_transform(mask)

        mask = remap_mask(mask)

        if self.transform:
            pass

        return image, mask


original_kagglehub_path = "/kaggle/input/q3-stage3-2026"
dataset_root = os.path.join(original_kagglehub_path, "dataset")

full_dataset = SUIMDataset(root_dir=dataset_root, image_folder="images", mask_folder="masks")

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print("DataLoaders created successfully.")

colors = [
    'black',
    'red',
    'green',
    'blue',
    'yellow',
    'cyan',
    'magenta',
    'white'
]
cmap = mcolors.ListedColormap(colors)
norm = mcolors.BoundaryNorm(boundaries=np.arange(-0.5, 8.5, 1), ncolors=8)

num_samples_to_display = 4
displayed_samples = []

for batch_idx, (current_images, current_masks) in enumerate(train_loader):
    for i in range(current_images.shape[0]):
        mask = current_masks[i]
        if torch.unique(mask).numel() > 1:
            displayed_samples.append((current_images[i], mask))
            if len(displayed_samples) >= num_samples_to_display:
                break
    if len(displayed_samples) >= num_samples_to_display:
        break

if not displayed_samples:
    print("Warning: Could not find any masks with diverse class labels in the training set.")
    if 'current_images' in locals() and 'current_masks' in locals():
        for i in range(min(num_samples_to_display, current_images.shape[0])):
            displayed_samples.append((current_images[i], current_masks[i]))
    else:
        print("Error: No samples available to display.")

fig, axes = plt.subplots(2, num_samples_to_display + 1, figsize=(num_samples_to_display * 4 + 2, 8),
                         gridspec_kw={'width_ratios': [1]*num_samples_to_display + [0.1]})

for idx, (image, mask) in enumerate(displayed_samples):
    ax_img = axes[0, idx]
    img_display = image.permute(1, 2, 0).cpu().numpy()
    ax_img.imshow(img_display)
    ax_img.set_title(f'Image {idx+1}')
    ax_img.axis('off')

    ax_mask = axes[1, idx]
    mask_display = mask.cpu().numpy()
    c = ax_mask.imshow(mask_display, cmap=cmap, norm=norm)
    ax_mask.set_title(f'Mask {idx+1}')
    ax_mask.axis('off')

    print(f"Unique values in displayed mask {idx+1}: {torch.unique(mask)}")

axes[0, -1].axis('off')

cbar = fig.colorbar(c, cax=axes[1, -1], ticks=np.arange(0, 8))
cbar.set_ticklabels([
    '0: Background',
    '1: Human divers',
    '2: Aquatic plants',
    '3: Wrecks',
    '4: Robots',
    '5: Reefs',
    '6: Fish',
    '7: Sea-floor'
])
cbar.set_label('Class ID')

plt.tight_layout()
plt.show()


In [ ]:
# TO DO

In [ ]:
!pip install segmentation_models_pytorch

In [ ]:
import segmentation_models_pytorch as smp

num_classes = 8

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=num_classes,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print(f"U-Net model with efficientnet-b1 encoder and {num_classes} classes instantiated successfully.")
print(f"Model moved to: {device}")

In [ ]:
import torch

input_height, input_width = 256, 256
batch_size = 4

dummy_input = torch.randn(batch_size, 3, input_height, input_width).to(device)

with torch.no_grad():
    output = model(dummy_input)

print(f"Dummy input shape: {dummy_input.shape}")
print(f"Model output shape: {output.shape}")
expected_output_shape = (batch_size, num_classes, input_height, input_width)
if output.shape == expected_output_shape:
    print("Output shape is as expected.")
else:
    print(f"Warning: Output shape mismatch. Expected {expected_output_shape}, got {output.shape}")

In [ ]:
# TO DO
def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    model.train()
    running_loss = 0.0

    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_fn(outputs, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(dataloader)



In [ ]:
def validate_one_epoch(model, dataloader, loss_fn, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = loss_fn(outputs, masks)

            running_loss += loss.item()

    return running_loss / len(dataloader)


In [ ]:
# TO DO
import torch.nn as nn
import torch.optim as optim

loss_fn = nn.CrossEntropyLoss()

learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print("Loss function and optimizer initialized successfully.")

In [ ]:
num_epochs = 20
train_losses = []
val_losses = []

print("Starting training...")

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    val_loss = validate_one_epoch(model, val_loader, loss_fn, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

print("Training complete.")

# loss curve plt
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss')
plt.plot(range(1, num_epochs + 1), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Curve')
plt.legend()
plt.grid(True)
plt.show()

print(f"Final Training Loss: {train_losses[-1]:.4f}")
print(f"Final Validation Loss: {val_losses[-1]:.4f}")

In [ ]:
# TO DO